# Exercise 10 — Attention Mechanisms

**Task:** given a length-10 sequence of integers in `[0, 9]`, decide **"are there more 2s than 4s?"**

A fully-connected net could brute-force this, but we force the model to *learn where to look*: it embeds each integer, computes an attention weight per position, and forms an answer as the attention-weighted sum of per-position values. The attention formula is

$$ v_i = \mathrm{softmax}_j\!\left(\frac{q_i \cdot k_j}{\sqrt{d}}\right)\, v_j $$

If it works, the attention will concentrate on the **2s and 4s** and ignore everything else.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

## Step 1 — Generating data

`make_batch(N)` returns `N` length-10 sequences of random integers in `[0, 9]` (`X` of shape `(N, 10)`, integer dtype for the embedding) and a binary label `y` of shape `(N, 1)` that is `1` when the sequence contains strictly more 2s than 4s.

In [ ]:
def make_batch(N):
    X = torch.randint(0, 10, (N, 10))                 # (N, 10) integer tokens
    n2 = (X == 2).sum(dim=1)
    n4 = (X == 4).sum(dim=1)
    y = (n2 > n4).float().reshape(N, 1)               # (N, 1)
    return X, y


X, y = make_batch(5)
print("X shape:", X.shape, "| y shape:", y.shape)
print(X)
print("more 2s than 4s?", y.ravel())

## Step 2 — Embedding the integers

`torch.nn.Embedding(10, embed_dim)` is a lookup table: each of the 10 possible integers maps to a learnable `embed_dim`-vector. Feeding an `(N, M)` integer tensor returns `(N, M, embed_dim)`.

In [ ]:
embed_dim = 5
emb = torch.nn.Embedding(10, embed_dim)

test = torch.randint(0, 10, (3, 10))   # (N=3, M=10)
out = emb(test)
print("input  (N, M):        ", tuple(test.shape))
print("output (N, M, embed): ", tuple(out.shape), "<- each token became a 5-vector")

## Step 3 — Extracting keys and values

From the embedded tokens we project **keys** (used to decide *how much* attention each position gets) and **values** (the per-position contribution to the answer). Here keys are `att_dim`-vectors and each value is a single scalar.

In [ ]:
att_dim = 5
WK = torch.nn.Linear(embed_dim, att_dim)   # embedded -> key vector
WV = torch.nn.Linear(embed_dim, 1)         # embedded -> scalar value

e = emb(test)                # (3, 10, embed_dim)
keys = WK(e)                 # (3, 10, att_dim)
vals = WV(e)                 # (3, 10, 1)
print("embedded:", tuple(e.shape))
print("keys:    ", tuple(keys.shape))
print("values:  ", tuple(vals.shape))

## Step 4 — Computing attention (batched, with `einsum`)

We have a **single** learnable query `q` of shape `(1, att_dim)`. For a batch of key sets `(N, 10, att_dim)` we want, for every sequence, the attention weight over its 10 positions:

1. score each position: `q · k_j / sqrt(att_dim)` → `(N, 1, 10)`
2. softmax over the 10 positions → attention weights that sum to 1

`torch.einsum('qk,njk->nqj', q, keys)` does the batched dot product: `n`=batch, `q`=query index (just 1), `j`=position, `k`=the vector coordinates being summed over.

In [ ]:
q = torch.randn(1, att_dim)                                  # single query
scores = torch.einsum('qk,njk->nqj', q, keys) / np.sqrt(att_dim)   # (N, 1, 10)
att = torch.softmax(scores, dim=-1)                          # (N, 1, 10)
print("scores:", tuple(scores.shape), "| attention:", tuple(att.shape))
print("each row sums to 1:", att.sum(dim=-1).ravel())

## Step 5 — Integrate into a Module

We complete the provided skeleton. `attention(x)` returns the `(N, 1, 10)` weights, `values(x)` returns the `(N, 10, 1)` per-position values, and `forward(x)` combines them into the attention-weighted value and pushes that scalar through a small MLP for the final probability.

The weighted value is `sum_j att_j * value_j`, again a batched contraction — `einsum('nqj,njv->nqv', att, values)` → `(N, 1, 1)`.

In [ ]:
class AttentionModel(torch.nn.Module):
    def __init__(self):
        super(AttentionModel, self).__init__()
        self.embed_dim = 5
        self.att_dim = 5
        self.embed = torch.nn.Embedding(10, self.embed_dim)

        # one query
        self.query = torch.nn.Parameter(torch.randn(1, self.att_dim))

        # used to compute keys
        self.WK = torch.nn.Linear(self.embed_dim, self.att_dim)

        # used to compute values
        self.WV = torch.nn.Linear(self.embed_dim, 1)

        # final decision based on attention-weighted value
        self.nn = torch.nn.Sequential(
            torch.nn.Linear(1, 200),
            torch.nn.ReLU(),
            torch.nn.Linear(200, 1),
            torch.nn.Sigmoid(),
        )

    def attention(self, x):
        # x: embedded tokens (N, 10, embed_dim)
        keys = self.WK(x)                                              # (N, 10, att_dim)
        scores = torch.einsum('qk,njk->nqj', self.query, keys)         # (N, 1, 10)
        scores = scores / np.sqrt(self.att_dim)
        return torch.softmax(scores, dim=-1)                           # (N, 1, 10)

    def values(self, x):
        # x: embedded tokens (N, 10, embed_dim)
        return self.WV(x)                                             # (N, 10, 1)

    def forward(self, x):
        e = self.embed(x)                                            # (N, 10, embed_dim)
        att = self.attention(e)                                      # (N, 1, 10)
        vals = self.values(e)                                        # (N, 10, 1)
        weighted = torch.einsum('nqj,njv->nqv', att, vals)           # (N, 1, 1)
        weighted = weighted.reshape(-1, 1)                           # (N, 1)
        return self.nn(weighted)                                     # (N, 1) probability


# shape sanity check
model = AttentionModel()
Xb, yb = make_batch(4)
print("forward output:", tuple(model(Xb).shape))
print("attention:     ", tuple(model.attention(model.embed(Xb)).shape))
print("values:        ", tuple(model.values(model.embed(Xb)).shape))

## Step 6 — Train and plot

Train on binary cross-entropy. Following the exercise's suggestion we use a large batch (500) and a small learning rate (5e-5). The task is subtle, so it needs a good number of steps.

In [ ]:
def train(model, steps=8000, N=500, lr=5e-5):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = torch.nn.BCELoss()
    traj = []
    for step in range(steps):
        X, y = make_batch(N)
        p = model(X)
        loss = loss_fn(p, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        traj.append(loss.item())
        if step % 500 == 0:
            print(f"step {step:5d}  loss {loss.item():.4f}")
    return traj


model = AttentionModel()
traj = train(model)

In [ ]:
# accuracy on a fresh batch
with torch.no_grad():
    Xt, yt = make_batch(2000)
    acc = ((model(Xt) > 0.5).float() == yt).float().mean().item()
print(f"test accuracy: {acc:.3f}")

### Visualizing the attention

The provided plotting function shows, for a batch of sequences: (left) the attention weight on each position, (middle) the value at the attended positions, (right) the loss trajectory. The digits **2 and 4 are drawn in red** — if learning worked, the bright attention cells line up with the red digits.

In [ ]:
def plot(model, N, traj):
    x, y = make_batch(N)
    f, axarr = plt.subplots(1, 3)
    f.set_size_inches(13, 2)
    ax = axarr[0]
    at = model.attention(model.embed(x))[:, 0, :].detach().numpy()
    ax.imshow(at)
    ax = axarr[1]

    vals = model.values(model.embed(x))[:, :, 0].detach().numpy()
    nan = np.ones_like(vals) * np.nan
    nan = np.where(at > 0.1, vals, nan)
    ax.imshow(nan, vmin=-1, vmax=1)
    for i, xx in enumerate(x):
        for j, xxx in enumerate(xx):
            ax = axarr[0]
            ax.text(j, i, xxx.numpy(), c='r' if (xxx in [2, 4]) else 'w')
            ax = axarr[1]
            ax.text(j, i, xxx.numpy(), c='r' if (xxx in [2, 4]) else 'w')
    ax = axarr[2]
    ax.plot(traj)


plot(model, 10, traj)
plt.show()

### What to point out in the presentation

* **The model learns *where to look*.** In the left panel the bright (high-attention) cells sit on the **red digits (2s and 4s)** and the other eight positions stay dark — even though nothing ever told the model which digits matter. It discovered that from the labels alone.
* **Why the middle panel matters:** the network learns *opposite-sign* values for 2s vs 4s, so the attention-weighted sum is positive when 2s dominate and negative when 4s do — that single scalar is all the final MLP needs.
* **Why `einsum`:** it expresses the batched dot-products (`q·k` and `att·v`) in one readable line, no manual reshaping/transposing.
* **Design choice:** attention decouples *what to attend to* (keys/query) from *what to report* (values) — the same idea that scales up to transformers, just with many queries and stacked layers.